# 6-2. 품목제조대장 유통기한(`pog_daycnt`) 조인과 결측 확인

- 작성: 2026-09-28
- 입력: `리스테리아ML/data/listeria_df_joined_full.parquet` (노트북 6-1 저장본, 676,537행 × 37열, 검출률 0% 품목 삭제 안 함)
- 붙일 값: `ba_data.t_prdlst_mnf_ledg`(품목제조대장, 정의서 37컬럼) 의 `pog_daycnt`(DB 코멘트 "유통기한", text) — 원문 그대로, 해석·변환·채움 없음
- **데이터 파일은 저장하지 않는다.** 이 노트북은 조인 과정과 결측 수치를 기록하는 용도다.

### 조인 기준 (사용자 승인 2026-09-28)
| 단계 | 기준 |
|---|---|
| 1. 품목제조번호 가져오기 | `lims_data_key` = `ready_data.hazard_preprocessed_data.lims_data_key`(`hazard_label = '리스테리아'`, 1:1) → `prdlst_mnfctur_sttemnt_no`. 이 파일에는 품목제조번호가 없어서 거쳐야 한다. |
| 2. 품목제조대장 붙이기 | `prdlst_mnfctur_sttemnt_no` = `ba_data.t_prdlst_mnf_ledg.prdlst_report_no`(DB 코멘트 "품목 보고 번호") **글자 그대로 같은 것만** (기호 제거·공백 제거 같은 변환 없음) |
| 같은 번호가 대장에 여러 행 | **연결하지 않고 결측으로 둔다** (어느 행을 고를지 정하는 규칙을 쓰지 않음). 결측 이유를 따로 센다. |
| 결과물 | 이 노트북 + 결측 보고. 데이터 파일 저장 없음 |

검토했지만 쓰지 않은 방식: (1) 번호가 원문으로 안 맞으면 '-'·'호' 같은 글자를 지운 숫자만으로 한 번 더 맞추기(2026-09-17 공유용 데이터셋 방식 — 이 변환이 식정원 번호 체계인지 확인 안 됨), (2) 대장 `org_prdlst_report_no`(변경전 품목보고번호)로 한 번 더 맞추기, (3) 같은 번호 여러 행이면 삭제 안 된 행 → 최신 `last_updt_dtm` → 큰 `prdlst_report_ledg_no` 순으로 1행 고르기(2026-09-17 공유용 규칙).

## 1. 입력 파일 읽기

In [1]:
import os

import numpy as np
import pandas as pd
import psycopg

IN_PATH = r"C:\Users\NT751\Desktop\천영진\리스테리아ML\data\listeria_df_joined_full.parquet"
df = pd.read_parquet(IN_PATH)

n_all = len(df)
n_pos = int((df["judge_result"] == "검출").sum())
print(f"행 {n_all:,} / 열 {df.shape[1]} / 검출 {n_pos:,}건 (검출률 {n_pos / n_all * 100:.3f}%)")
print("src_route 별 행 수:", df["src_route"].value_counts().to_dict())

행 676,537 / 열 37 / 검출 1,969건 (검출률 0.291%)
src_route 별 행 수: {'domestic': 628895, 'overseas': 47642}


## 2. DB 읽기 전용 접속과 쓰는 컬럼의 DB 코멘트

- 세션은 읽기 전용(`default_transaction_read_only=on`)으로 열고 SELECT 만 한다. 비밀번호는 `%APPDATA%\postgresql\pgpass.conf` 에서 libpq 가 읽는다(코드에 쓰지 않음).
- 큰 결과는 서버측 커서(named cursor)로 10만 행씩 나눠 받는다.

In [2]:
PGPASS = os.path.join(os.environ["APPDATA"], "postgresql", "pgpass.conf")   # 비밀번호는 libpq 가 이 파일에서 읽음
DB_HOST = "10.31.72.23"


def db_connect_readonly(autocommit=True):
    """pgpass.conf 의 host 항목에서 port·dbname·user 만 읽어 읽기 전용 세션으로 접속한다."""
    with open(PGPASS, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.replace("\\:", "\x00").split(":")
            if len(parts) < 5:
                continue
            host, port, dbname, user = (x.replace("\x00", ":") for x in parts[:4])
            if host in (DB_HOST, "*"):
                conn = psycopg.connect(
                    host=DB_HOST, port=5432 if port == "*" else int(port), dbname=dbname, user=user,
                    options="-c default_transaction_read_only=on -c statement_timeout=300000",
                    application_name="listeria_nb6_2", connect_timeout=15)
                conn.autocommit = autocommit
                return conn
    raise RuntimeError(f"pgpass.conf 에 {DB_HOST} 항목이 없습니다: {PGPASS}")


def db_query(sql, params=None):
    """작은 SELECT 결과를 DataFrame 으로."""
    with db_connect_readonly() as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


def db_fetch(sql, params=None, chunk=100_000):
    """큰 SELECT 결과를 서버측 커서로 나눠 받아 DataFrame 으로 (트랜잭션 블록 안, 읽기 전용)."""
    rows, cols = [], None
    with db_connect_readonly(autocommit=False) as conn:
        with conn.transaction():
            with conn.cursor(name="nb6_2_fetch") as cur:
                cur.execute(sql, params)
                while True:
                    part = cur.fetchmany(chunk)
                    if cols is None and cur.description:
                        cols = [d.name for d in cur.description]
                    if not part:
                        break
                    rows.extend(part)
    return pd.DataFrame(rows, columns=cols)


print("읽기 전용 세션:", db_query("show default_transaction_read_only").iloc[0, 0])
display(db_query("""
    select c.relnamespace::regnamespace::text || '.' || c.relname as 테이블, a.attname as 컬럼,
           format_type(a.atttypid, a.atttypmod) as 형식, col_description(a.attrelid, a.attnum) as db_코멘트
    from pg_attribute a join pg_class c on c.oid = a.attrelid
    where (c.oid = 'ready_data.hazard_preprocessed_data'::regclass and a.attname in ('lims_data_key', 'prdlst_mnfctur_sttemnt_no'))
       or (c.oid = 'ba_data.m_org_lims_data_info'::regclass and a.attname in ('lims_data_key', 'prdlst_mnfctur_sttemnt_no'))
       or (c.oid = 'ba_data.t_prdlst_mnf_ledg'::regclass
           and a.attname in ('prdlst_report_ledg_no', 'prdlst_report_no', 'pog_daycnt', 'pog_cn', 'del_yn', 'last_updt_dtm'))
    order by 1, a.attnum
"""))

읽기 전용 세션: on


,테이블,컬럼,형식,db_코멘트
0,ba_data.m_org_lims_data_info,lims_data_key,character varying(100),LIMS 데이터 키 (검사기관코드_시료의뢰번호_시료순번_시험항목순번)
1,ba_data.m_org_lims_data_info,prdlst_mnfctur_sttemnt_no,character varying(30),품목 제조 신고 번호
2,ba_data.t_prdlst_mnf_ledg,prdlst_report_ledg_no,text,품목 보고 대장 번호
3,ba_data.t_prdlst_mnf_ledg,prdlst_report_no,text,품목 보고 번호
4,ba_data.t_prdlst_mnf_ledg,pog_daycnt,text,유통기한
5,ba_data.t_prdlst_mnf_ledg,del_yn,text,삭제 여부
6,ba_data.t_prdlst_mnf_ledg,last_updt_dtm,timestamp without time zone,최종 수정 일시
7,ba_data.t_prdlst_mnf_ledg,pog_cn,text,"(사용중지, pog_daycnt로 단일화) 유통기한"
8,ready_data.hazard_preprocessed_data,lims_data_key,character varying(100),NaN
9,ready_data.hazard_preprocessed_data,prdlst_mnfctur_sttemnt_no,character varying(30),NaN


- `ready_data.hazard_preprocessed_data` 는 DB 코멘트가 비어 있다. 테이블정의서(`테이블명세서(모델팀)/테이블정의서.xlsx` ready_data 시트)의 `prdlst_mnfctur_sttemnt_no` 설명: "품목제조번호. 주소 체인 2순위(인허가대장) 연결 키. 수입·농관원은 NULL".
- `pog_cn` 은 DB 코멘트가 "(사용중지, pog_daycnt로 단일화) 유통기한"이라 쓰지 않는다.

## 3. 1단계 — `lims_data_key` 로 품목제조번호(`prdlst_mnfctur_sttemnt_no`) 가져오기

In [3]:
hp = db_fetch("""
    select lims_data_key::text as lims_data_key, prdlst_mnfctur_sttemnt_no::text as prdlst_mnfctur_sttemnt_no
    from ready_data.hazard_preprocessed_data
    where hazard_label = '리스테리아'
""")
print(f"ready_data.hazard_preprocessed_data (hazard_label='리스테리아'): {len(hp):,}행 / lims_data_key 고유 {hp['lims_data_key'].nunique():,}")
assert hp["lims_data_key"].is_unique, "lims_data_key 가 중복됩니다 — 1:1 연결이 아님"

dfl = df.copy()                                   # 이 노트북 안에서만 쓰는 작업본 (저장하지 않음)
key = dfl["lims_data_key"].astype(str)
no_map = hp.set_index("lims_data_key")["prdlst_mnfctur_sttemnt_no"]
dfl["prdlst_mnfctur_sttemnt_no"] = key.map(no_map)
print(f"파일 키 {len(dfl):,}개 중 ready_data 에 있음 {int(key.isin(no_map.index).sum()):,}개")

has_no = dfl["prdlst_mnfctur_sttemnt_no"].notna()
no_txt = dfl.loc[has_no, "prdlst_mnfctur_sttemnt_no"]
n_blank = int(no_txt.str.strip().eq("").sum())
n_nondigit = int(no_txt.str.contains(r"\D", regex=True).sum())
print(f"품목제조번호 있음 {int(has_no.sum()):,}행 ({has_no.mean() * 100:.2f}%) / 그중 빈 문자열·공백 {n_blank:,}행 / "
      f"고유 번호 {no_txt.nunique():,}개")
print(f"숫자 아닌 글자가 섞인 번호: {n_nondigit:,}행 (원문 일치만 쓰므로 변환하지 않음)")

t = (dfl.assign(번호있음=has_no)
        .groupby(["src_route", "judge_result"], observed=True)["번호있음"]
        .agg(행수="size", 번호있음="sum"))
t["번호있음(%)"] = (t["번호있음"] / t["행수"] * 100).round(2)
display(t)

ready_data.hazard_preprocessed_data (hazard_label='리스테리아'): 765,891행 / lims_data_key 고유 765,891


파일 키 676,537개 중 ready_data 에 있음 676,537개
품목제조번호 있음 204,115행 (30.17%) / 그중 빈 문자열·공백 0행 / 고유 번호 35,504개
숫자 아닌 글자가 섞인 번호: 844행 (원문 일치만 쓰므로 변환하지 않음)


행수    번호있음  번호있음(%)
src_route judge_result                         
domestic  검출              1844     225    12.20
          불검출           627051  203890    32.52
overseas  검출               125       0     0.00
          불검출            47517       0     0.00

In [4]:
# 참고: 같은 번호가 식약처 LIMS 원본(ba_data.m_org_lims_data_info)에도 있는지, 값이 같은지
mo = db_fetch("""
    select m.lims_data_key::text as lims_data_key, m.prdlst_mnfctur_sttemnt_no::text as mo_no
    from ba_data.m_org_lims_data_info m
    where m.lims_data_key in (select lims_data_key from ready_data.hazard_preprocessed_data where hazard_label = '리스테리아')
""")
print(f"ba_data.m_org_lims_data_info 행 {len(mo):,} / lims_data_key 고유 {mo['lims_data_key'].nunique():,}")
cmp_ = dfl[["lims_data_key", "src_route", "prdlst_mnfctur_sttemnt_no"]].assign(lims_data_key=key).merge(
    mo.drop_duplicates("lims_data_key"), on="lims_data_key", how="left", indicator=True)
in_mo = cmp_["_merge"] == "both"
same = cmp_["prdlst_mnfctur_sttemnt_no"].fillna("<NA>") == cmp_["mo_no"].fillna("<NA>")
print(f"파일 행 중 m_org 에 있음 {int(in_mo.sum()):,}행 → 번호 같음(둘 다 비어 있는 경우 포함) {int((in_mo & same).sum()):,} / 다름 {int((in_mo & ~same).sum()):,}")
print("m_org 에 없는 파일 행의 src_route:", cmp_.loc[~in_mo, "src_route"].astype(str).value_counts().to_dict())

ba_data.m_org_lims_data_info 행 758,574 / lims_data_key 고유 758,574


파일 행 중 m_org 에 있음 669,334행 → 번호 같음(둘 다 비어 있는 경우 포함) 669,334 / 다름 0
m_org 에 없는 파일 행의 src_route: {'overseas': 7203}


**결과 (1단계)**
- 파일의 676,537개 키가 `ready_data.hazard_preprocessed_data` 에 모두 있고 1:1 이다(키 중복 0).
- 품목제조번호가 있는 행은 204,115행(전체의 30.17%, 고유 번호 35,504개)이고 모두 국내 행이다. 수입 47,642행은 번호가 전부 비어 있다(정의서 설명 "수입·농관원은 NULL"과 같음).
- 국내 행에서 번호가 있는 비율: 검출 12.20%(225/1,844) · 불검출 32.52%(203,890/627,051).
- 식약처 LIMS 원본 `ba_data.m_org_lims_data_info.prdlst_mnfctur_sttemnt_no` 와 비교하면, 파일 행 중 m_org 에 있는 669,334행은 값이 전부 같다. m_org 에 없는 7,203행은 모두 수입 행이다. 따라서 번호를 어느 테이블에서 가져와도 결과는 같다.

## 4. 2단계 — 품목제조대장(`ba_data.t_prdlst_mnf_ledg`)에서 같은 번호의 행 가져오기

파일에 있는 품목제조번호(원문)만 넘겨서 `prdlst_report_no` 가 **글자 그대로 같은** 대장 행을 모두 가져온다. 번호마다 대장 행이 몇 개인지(0 / 1 / 2개 이상) 센다.

In [5]:
nos = sorted(dfl.loc[has_no, "prdlst_mnfctur_sttemnt_no"].unique().tolist())
ledg = db_fetch("""
    select prdlst_report_ledg_no, prdlst_report_no, pog_daycnt, del_yn, last_updt_dtm, prdlst_nm, prms_dt
    from ba_data.t_prdlst_mnf_ledg
    where prdlst_report_no = any(%s::text[])
""", (nos,))
print(f"넘긴 고유 번호 {len(nos):,}개 → 대장 행 {len(ledg):,}개 (번호 {ledg['prdlst_report_no'].nunique():,}개)")

n_per_no = ledg.groupby("prdlst_report_no").size()
k = pd.Series(nos).map(n_per_no).fillna(0).astype(int)
print(f"고유 번호별 대장 행 수: 0행(대장에 없음) {int((k == 0).sum()):,}개 / 1행 {int((k == 1).sum()):,}개 / "
      f"2행 이상 {int((k >= 2).sum()):,}개")
print("대장 행 수가 가장 많은 번호 상위:", n_per_no.sort_values(ascending=False).head(5).to_dict())

넘긴 고유 번호 35,504개 → 대장 행 35,808개 (번호 35,089개)
고유 번호별 대장 행 수: 0행(대장에 없음) 415개 / 1행 34,401개 / 2행 이상 688개
대장 행 수가 가장 많은 번호 상위: {'2016014708544': 4, '2016014708554': 4, '201601740253': 4, '2009020255329': 3, '2010026209446': 3}


**결과 (2단계)**: 고유 번호 35,504개 중 대장에 정확히 1행 있는 번호 34,401개, 2행 이상 688개(2행 660 · 3행 25 · 4행 3), 대장에 없는 번호 415개.

## 5. 연결과 결측 이유 나누기

- 대장 행이 **정확히 1행**인 번호만 `pog_daycnt` 를 붙인다(그 행의 `del_yn` 과 상관없이 — 삭제 행을 뺄지는 정하지 않았으므로 7절에 따로 센다).
- 결측 이유(행마다 하나):
  - ① 품목제조번호 없음 — 1단계 `prdlst_mnfctur_sttemnt_no` 가 NULL
  - ② 번호가 대장에 없음 — 같은 `prdlst_report_no` 대장 행 0개
  - ③ 대장에 같은 번호 여러 행 — 2행 이상이라 연결하지 않음(승인된 규칙)
  - ④ 연결됐으나 `pog_daycnt` 가 NULL
  - ⑤ 연결됐으나 `pog_daycnt` 가 빈 문자열이거나 공백만 있음
  - 값 있음 — 위 어디에도 해당하지 않음(값의 뜻은 해석하지 않음)

In [6]:
n_match = dfl["prdlst_mnfctur_sttemnt_no"].map(n_per_no).fillna(0).astype(int)
single = ledg[ledg["prdlst_report_no"].isin(n_per_no[n_per_no == 1].index)].set_index("prdlst_report_no")

dfl["ledg_n_rows"] = n_match.where(has_no).astype("Int64")                   # 번호별 대장 행 수 (번호 없으면 비움)
dfl["pog_daycnt"] = dfl["prdlst_mnfctur_sttemnt_no"].map(single["pog_daycnt"])  # 대장 1행인 번호만 붙음
dfl["ledg_del_yn"] = dfl["prdlst_mnfctur_sttemnt_no"].map(single["del_yn"])      # 참고(7절)

pog = dfl["pog_daycnt"]
REASONS = ["① 품목제조번호 없음", "② 번호가 대장에 없음", "③ 대장에 같은 번호 여러 행(연결 안 함)",
           "④ 연결됐으나 pog_daycnt NULL", "⑤ 연결됐으나 빈 문자열·공백", "값 있음"]
dfl["pog_status"] = pd.Categorical(np.select(
    [~has_no.to_numpy(), (n_match == 0).to_numpy(), (n_match >= 2).to_numpy(),
     pog.isna().to_numpy(), pog.fillna("x").str.strip().eq("").to_numpy()],
    REASONS[:5], default=REASONS[5]), categories=REASONS)

assert len(dfl) == n_all
linked = dfl["pog_status"].isin(REASONS[3:])
print(f"대장 1행 연결 {int(linked.sum()):,}행 / pog_daycnt 값 있음 {int((dfl['pog_status'] == '값 있음').sum()):,}행 "
      f"/ 결측 {int((dfl['pog_status'] != '값 있음').sum()):,}행 ({(dfl['pog_status'] != '값 있음').mean() * 100:.2f}%)")

대장 1행 연결 200,156행 / pog_daycnt 값 있음 191,631행 / 결측 484,906행 (71.67%)


## 6. `pog_daycnt` 결측 요약

- 결측률(%) = 결측 행 / 그 그룹 행 수 × 100 (결측 = '값 있음'이 아닌 모든 행)
- 이유별 비율은 그 그룹 행 수 대비

In [7]:
def label(s, na="(없음)"):
    """그룹 이름용 — 결측도 한 그룹으로 보이게 (표시용, 데이터는 바꾸지 않음)."""
    return s.astype("string").fillna(na)


def status_table(by=None):
    """by 그룹별 결측 이유 행 수 + 결측률. by=None 이면 전체."""
    g = dfl.assign(_all="전체") if by is None else dfl
    keys = ["_all"] if by is None else by
    ct = pd.crosstab([label(g[k], "(train_prdlst_code 없음)") for k in keys], g["pog_status"])
    ct = ct.reindex(columns=REASONS, fill_value=0)
    ct.insert(0, "행수", ct.sum(axis=1))
    ct["결측"] = ct["행수"] - ct["값 있음"]
    ct["결측률(%)"] = (ct["결측"] / ct["행수"] * 100).round(2)
    return ct


print("[전체]")
tot = status_table()
display(tot)
share = (tot[REASONS].iloc[0] / n_all * 100).round(2).rename("전체 행 대비(%)")
display(share.to_frame())

[전체]


pog_status,행수,① 품목제조번호 없음,② 번호가 대장에 없음,③ 대장에 같은 번호 여러 행(연결 안 함),④ 연결됐으나 pog_daycnt NULL,⑤ 연결됐으나 빈 문자열·공백,값 있음,결측,결측률(%)
_all,,,,,,,,,
전체,676537,472422,1189,2770,8525,0,191631,484906,71.67


,전체 행 대비(%)
pog_status,
① 품목제조번호 없음,69.83
② 번호가 대장에 없음,0.18
③ 대장에 같은 번호 여러 행(연결 안 함),0.41
④ 연결됐으나 pog_daycnt NULL,1.26
⑤ 연결됐으나 빈 문자열·공백,0.00
값 있음,28.33


In [8]:
print("[국내/수입 — src_route]")
display(status_table(["src_route"]))
print("[검출/불검출 — judge_result]")
display(status_table(["judge_result"]))
print("[국내 행만 — src_route × judge_result]")
display(status_table(["src_route", "judge_result"]).loc[["domestic"]])

[국내/수입 — src_route]


pog_status,행수,① 품목제조번호 없음,② 번호가 대장에 없음,③ 대장에 같은 번호 여러 행(연결 안 함),④ 연결됐으나 pog_daycnt NULL,⑤ 연결됐으나 빈 문자열·공백,값 있음,결측,결측률(%)
src_route,,,,,,,,,
domestic,628895,424780,1189,2770,8525,0,191631,437264,69.53
overseas,47642,47642,0,0,0,0,0,47642,100.00


[검출/불검출 — judge_result]


pog_status,행수,① 품목제조번호 없음,② 번호가 대장에 없음,③ 대장에 같은 번호 여러 행(연결 안 함),④ 연결됐으나 pog_daycnt NULL,⑤ 연결됐으나 빈 문자열·공백,값 있음,결측,결측률(%)
judge_result,,,,,,,,,
검출,1969,1744,1,3,0,0,221,1748,88.78
불검출,674568,470678,1188,2767,8525,0,191410,483158,71.62


[국내 행만 — src_route × judge_result]


pog_status                  행수  ① 품목제조번호 없음  ② 번호가 대장에 없음  \
src_route judge_result                                      
domestic  검출              1844         1619             1   
          불검출           627051       423161          1188   

pog_status              ③ 대장에 같은 번호 여러 행(연결 안 함)  ④ 연결됐으나 pog_daycnt NULL  \
src_route judge_result                                                      
domestic  검출                                   3                        0   
          불검출                               2767                     8525   

pog_status              ⑤ 연결됐으나 빈 문자열·공백    값 있음      결측  결측률(%)  
src_route judge_result                                            
domestic  검출                           0     221    1623   88.02  
          불검출                          0  191410  435641   69.47

In [9]:
print("[최상위 분류 — lv1_nm (단계 조인이 안 된 '기타' 170행은 '(train_prdlst_code 없음)')]")
lv = status_table(["lv1_nm"])
is_pos = dfl["judge_result"].eq("검출")
pos_val = (pd.DataFrame({"lv1_nm": label(dfl["lv1_nm"], "(train_prdlst_code 없음)"),
                         "검출": is_pos, "검출 중 값 있음": is_pos & dfl["pog_status"].eq("값 있음")})
             .groupby("lv1_nm")[["검출", "검출 중 값 있음"]].sum())
display(lv.join(pos_val).sort_values("행수", ascending=False))

[최상위 분류 — lv1_nm (단계 조인이 안 된 '기타' 170행은 '(train_prdlst_code 없음)')]


,행수,① 품목제조번호 없음,② 번호가 대장에 없음,③ 대장에 같은 번호 여러 행(연결 안 함),④ 연결됐으나 pog_daycnt NULL,⑤ 연결됐으나 빈 문자열·공백,값 있음,결측,결측률(%),검출,검출 중 값 있음
lv1_nm,,,,,,,,,,,
가공식품,333276,130792,1133,2738,8506,0,190107,143169,42.96,449,220
기준규격외,320550,319308,44,28,18,0,1152,319398,99.64,1431,1
동물성,10204,10169,0,0,0,0,35,10169,99.66,74,0
식품접객업소,9654,9625,0,0,1,0,28,9626,99.71,0,0
식물성,2098,2086,12,0,0,0,0,2098,100.00,15,0
건강기능식품,446,152,0,0,0,0,294,152,34.08,0,0
(train_prdlst_code 없음),170,163,0,1,0,0,6,164,96.47,0,0
첨가물,139,127,0,3,0,0,9,130,93.53,0,0


**결과 — `pog_daycnt` 결측 484,906행 / 676,537행 (71.67%)**

| 결측 이유 | 행 수 | 전체 행 대비 |
|---|---:|---:|
| ① 품목제조번호 없음 | 472,422 | 69.83% |
| ② 번호가 대장에 없음 | 1,189 | 0.18% |
| ③ 대장에 같은 번호 여러 행(연결 안 함) | 2,770 | 0.41% |
| ④ 연결됐으나 `pog_daycnt` NULL | 8,525 | 1.26% |
| ⑤ 연결됐으나 빈 문자열·공백 | 0 | 0.00% |
| **결측 합계** | **484,906** | **71.67%** |
| 값 있음 | 191,631 | 28.33% |

- 결측의 97.4%(472,422 / 484,906)는 품목제조번호가 처음부터 없는 행(①)이다. 번호가 있는 204,115행만 보면 값 있음은 191,631행(93.9%)이다.
- 수입 47,642행은 100% 결측(전부 ①)이고, 국내 결측률은 69.53%다.
- 검출/불검출: 검출 1,969행 중 값 있음 221행으로 결측률 88.78%, 불검출은 71.62%. 국내 행만 보면 검출 88.02%, 불검출 69.47%.
- 최상위 분류(`lv1_nm`)별 결측률: 건강기능식품 34.08% · 가공식품 42.96% · 첨가물 93.53% · 기준규격외 99.64% · 동물성 99.66% · 식품접객업소 99.71% · 식물성 100%. 값 있음 191,631행 중 190,107행(99.2%)이 가공식품이다.
- 값이 있는 검출 행 221행은 가공식품 220행, 기준규격외 1행이다. 가공식품 검출 449건 중 220건(49.0%)에 값이 있고, 기준규격외 검출 1,431건 중에는 1건이다.

## 7. 참고 — ③ 여러 행 번호의 상태, 삭제 행 연결, `pog_daycnt` 값 예시

아래는 결정 자료로만 보여 주며 규칙을 바꾸지 않는다.

In [10]:
# ③ 대장에 같은 번호 여러 행 — 후보 행끼리 pog_daycnt 가 같은지, 삭제 안 된 행이 몇 개인지
multi_nos = n_per_no[n_per_no >= 2].index
lm = ledg[ledg["prdlst_report_no"].isin(multi_nos)].copy()
lm["_live"] = lm["del_yn"].fillna("N") != "Y"
per = lm.groupby("prdlst_report_no").agg(
    행수=("prdlst_report_ledg_no", "size"),
    pog_값_종류=("pog_daycnt", lambda s: s.fillna("<NULL>").nunique()),
    삭제안된행=("_live", "sum"))
rows3 = dfl[dfl["pog_status"] == REASONS[2]]
r3 = rows3["prdlst_mnfctur_sttemnt_no"].map(per["pog_값_종류"])
l3 = rows3["prdlst_mnfctur_sttemnt_no"].map(per["삭제안된행"])
print(f"③ 해당: 고유 번호 {len(multi_nos):,}개 / 시료 {len(rows3):,}행 (검출 {int((rows3['judge_result'] == '검출').sum()):,}행)")
print(f"  - 후보 행끼리 pog_daycnt 가 모두 같은 번호 {int((per['pog_값_종류'] == 1).sum()):,}개 → 시료 {int((r3 == 1).sum()):,}행")
print(f"  - 후보 행끼리 pog_daycnt 가 다른 번호   {int((per['pog_값_종류'] > 1).sum()):,}개 → 시료 {int((r3 > 1).sum()):,}행")
print(f"  - 삭제 안 된 행(del_yn≠'Y')이 정확히 1개인 번호 {int((per['삭제안된행'] == 1).sum()):,}개 → 시료 {int((l3 == 1).sum()):,}행")
print("  - 번호당 대장 행 수 분포:", per["행수"].value_counts().sort_index().to_dict())

③ 해당: 고유 번호 688개 / 시료 2,770행 (검출 3행)
  - 후보 행끼리 pog_daycnt 가 모두 같은 번호 490개 → 시료 1,744행
  - 후보 행끼리 pog_daycnt 가 다른 번호   198개 → 시료 1,026행
  - 삭제 안 된 행(del_yn≠'Y')이 정확히 1개인 번호 151개 → 시료 912행
  - 번호당 대장 행 수 분포: {2: 660, 3: 25, 4: 3}


In [11]:
# 대장 1행 연결인데 그 행이 삭제 표시(del_yn = 'Y')인 경우 — 제외하지 않았음(정하지 않음)
print("대장 1행 연결 행의 del_yn:", dfl.loc[linked, "ledg_del_yn"].fillna("<NULL>").value_counts().to_dict())

# ② 번호가 대장에 없음 — 번호 모양
nf = dfl.loc[dfl["pog_status"] == REASONS[1], "prdlst_mnfctur_sttemnt_no"]
nf_nondigit = int(nf.str.contains(r"\D", regex=True).sum())
print(f"② 해당: 시료 {len(nf):,}행 / 고유 번호 {nf.nunique():,}개 / 숫자 아닌 글자가 섞인 번호의 행 {nf_nondigit:,}행")
print("② 번호 길이(글자 수) 분포:", nf.str.len().value_counts().sort_index().to_dict())
display(nf.value_counts().head(15).rename_axis("② prdlst_mnfctur_sttemnt_no (원문)").reset_index(name="행수"))

대장 1행 연결 행의 del_yn: {'N': 200089, 'Y': 67}
② 해당: 시료 1,189행 / 고유 번호 415개 / 숫자 아닌 글자가 섞인 번호의 행 844행
② 번호 길이(글자 수) 분포: {1: 45, 2: 1, 4: 2, 5: 1, 6: 12, 8: 1, 9: 56, 10: 4, 11: 52, 12: 24, 13: 200, 14: 384, 15: 371, 16: 36}


,② prdlst_mnfctur_sttemnt_no (원문),행수
0,즉석판매제조가공업,51
1,-,45
2,2006040501402,27
3,20110476011-5,12
4,20110476011-2,12
5,20110405011-10,12
6,20110405011-1,12
7,20110405011-18,12
8,20110476011-8,12
9,19980407601676,12


- 대장 1행 연결 200,156행 중 67행은 그 대장 행이 삭제 표시(`del_yn = 'Y'`)다. 빼는 규칙은 정하지 않아 그대로 붙였다.
- ② 번호가 대장에 없는 1,189행(고유 번호 415개) 중 844행은 번호에 숫자 아닌 글자가 섞여 있다. 하이픈이 든 번호(예: `20110476011-5`)와 번호가 아닌 글자(`즉석판매제조가공업` 51행, `-` 45행)가 여기에 들어간다. 나머지 345행은 숫자만 있는 번호인데 대장에 같은 번호가 없다(예: `2006040501402` 27행). 원문 일치만 쓰므로 모두 연결하지 않았다.

In [12]:
# pog_daycnt 값 예시 — 원문 그대로, 해석하지 않음
v = dfl.loc[dfl["pog_status"] == "값 있음", "pog_daycnt"]
print(f"값 있음 {len(v):,}행 / 고유 값 {v.nunique():,}개")
with pd.option_context("display.max_colwidth", 80):
    display(v.value_counts().head(25).rename_axis("pog_daycnt").reset_index(name="행수"))

값 있음 191,631행 / 고유 값 6,984개


,pog_daycnt,행수
0,제조일로부터 12개월,5505
1,제조일로부터 6개월,3471
2,제조일로부터 60일,3372
3,제조일로부터 12개월까지,3199
4,제조일로부터 30일,2458
5,제조일로부터 1년,2324
6,제조일로부터 20일,2264
7,제조일로부터 3개월,2220
8,12개월,1718
9,제조일로부터 45일,1639


## 8. 정리

- **결측 71.67%**(484,906 / 676,537행). 대부분(97.4%)은 품목제조번호가 없는 행이다. 수입 행은 전부, 국내는 67.5%(424,780 / 628,895)가 번호 없음이다.
- 값이 있는 191,631행의 99.2%가 가공식품이다. 가공식품만 보면 결측률은 42.96%다.
- 값은 `제조일로부터 12개월`, `제조시간으로부터 48시간까지` 같은 문장이고 고유 값이 6,984개다. 일수 숫자가 아니므로, 모델 변수로 쓰려면 문장을 일수로 바꾸는 규칙이 따로 필요하다(바꾸는 규칙은 사용자 승인 뒤에 정함).

**남은 질문**
1. 결측이 타깃과 얽혀 있다. 검출 행 결측률은 88.78%이고 불검출은 71.62%다. 값이 있는 검출 행은 거의 모두 가공식품이다(221행 중 220행). 이 컬럼을 넣으면 '대장에 연결되는 행인지'가 가공식품 여부나 검사 경로를 대신 나타낼 수 있다(추정 — 확인 필요). 전체 행에 넣을지, 가공식품 행에만 쓸지 정해야 한다.
2. ③ 대장에 같은 번호가 여러 행인 2,770행 중, 후보 행끼리 `pog_daycnt` 가 모두 같은 번호가 490개(시료 1,744행)다. 이 경우는 연결할지 정해야 한다(지금은 승인된 규칙대로 전부 결측).
3. 삭제 표시(`del_yn = 'Y'`) 대장 행에서 온 값 67행을 그대로 둘지 정해야 한다.
4. 대장에는 수정 이력이 없어 지금 시점의 값이다. 시료 검사 당시의 유통기한과 다를 수 있다.
5. ② 번호 칸의 `즉석판매제조가공업`·`-`, 하이픈 번호는 원자료 입력 형식이다. 이런 값이 왜 들어갔는지 모델팀에 물어볼지 정해야 한다.